# The Gradio app — one UI over the same pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cwinkelmann/GEE_Animation/blob/main/notebooks/02_gradio_app.ipynb)

`gee-animation-gui` is not a separate product: it is one function, `build_app()`,
laid over exactly the `RunConfig` the CLI reads from YAML. Everything you click
maps onto a config key, and everything the config can say, a notebook or script
can say too.

There are three ways to run it — in order of how much you should trust them.

## 1 · Locally (the way you should run it)

Once, in a terminal:

```bash
pip install -e ".[gui,shapefile]"    # gradio (+ geopandas for shapefile uploads)
earthengine authenticate             # one-time, under YOUR project
EE_PROJECT=<your-project-id> gee-animation-gui   # or: python -m gee_animation.gui
```

What you get in the browser:

- **Project**: pre-filled from `EE_PROJECT`. If the variable is unset, the form
  falls back to the **operator's** project id — your renders would bill them —
  so set it, and check the box before you generate.
- **AOI**: upload a GeoJSON or zipped shapefile, or pick a **site preset** —
  curated site · edition combinations defined in `gee_animation/presets.py`
  (their footprints ship in `gee_animation/presets_aoi/`). **Careful:** *Apply
  preset* fills the whole form with the site's values, *including the Project
  field* — set it back to your own before you generate.
- **Sensor and index** pickers (the index choices update per sensor), date range,
  cloud threshold, resolution preset.
- Two accordions expose the thermal work: *Thermal sharpening & region focus*
  and *Seasonal model & anomaly*. They map one-to-one onto the config keys
  `sharpen`, `region_only`, `relative`, `render.pixel_grid`, `render.upscale`,
  `smooth`/`harmonics`, `anomaly`/`baseline_years` — the same keys notebooks 03
  and 04 teach.

Outputs land in `out/`; one render runs at a time, and the MP4, GIF, frame
gallery and the inside/outside chart appear inline when it finishes.

## 2 · On carrot (the deployment that exists)

The app has run on HNEE's `carrot` server since 2026-09-30 — image
`gee-timelapse`, container `gee-animation`. This is the recipe that produced it
(from a workstation checkout; `carrot` is an SSH alias for the host):

```bash
rsync -av --exclude .git --exclude 'out/' --exclude '*.mp4' . carrot:~/GEE_animation/
ssh carrot "cd ~/GEE_animation && docker build -t gee-timelapse ."
ssh carrot "docker rm -f gee-animation 2>/dev/null; docker run -d --name gee-animation \
  -p 7860:7860 \
  -v /raid/cwinkelmann/w/gee_out:/app/out \
  -v /raid/cwinkelmann/w/gee_cache:/cache \
  -v ~/keys:/keys:ro -v ~/GEE_animation/docs/aoi/wne/wne.geojson:/data/aoi.geojson:ro \
  -e EE_SERVICE_ACCOUNT_KEY=/keys/ee-key.json -e EE_PROJECT=hnee-331218 \
  -e GEE_ANIMATION_CACHE_DIR=/cache -e GEE_OUTPUT_DIR=/app/out \
  -e GEE_DEFAULT_AOI=/data/aoi.geojson gee-timelapse"
```

Be clear about what this is: a **single-operator deployment**. It is bound to
`0.0.0.0:7860` on the HNEE VPN, there is **no authentication**, and every render
runs under **one shared Earth Engine identity** (the operator's service account,
project `hnee-331218`) — whoever opens the page renders on that quota and sees
everyone's runs. For a trusted operator that is a fine tool; for a class it is
not. **Students: look, then render locally under your own project (§1).**

The multi-user web app in this repo ([docs/multiuser-server.md](../docs/multiuser-server.md))
was built to fix exactly this — per-user sign-in, per-user projects and quotas —
and awaits its first deployment.

## 3 · In Colab (a demo, not a deployment)

Yes, it works here too. Gradio detects Colab and prints a public
`*.gradio.live` link; the backend is your Colab VM and your Earth Engine
project, and it dies with the runtime. Don't build on it — but seeing the whole
app appear out of one function call is worth thirty seconds.

In [ ]:
import sys
IS_COLAB = "google.colab" in sys.modules
BRANCH = "main"
URL = f"gee_animation[gui,shapefile] @ git+https://github.com/cwinkelmann/GEE_Animation.git@{BRANCH}"
if IS_COLAB:
    !pip install -q "{URL}"                              # deps + package (first run)
    !pip install -q --force-reinstall --no-deps "{URL}"  # branch moved? force-refresh our code
# pip resolves by version (0.2.0), which doesn't change per commit on a branch --
# --force-reinstall pulls the latest commit. After a code update: Runtime ->
# Restart session, then Run all, so the new module is re-imported.

In [ ]:
import os
PROJECT = os.environ.get("EE_PROJECT", "")   # <-- or paste your own project id here
assert PROJECT, "Set PROJECT to your own Earth Engine Cloud project id — 00_setup.ipynb shows how to create one"

from gee_animation import auth
auth.init(PROJECT)

In [ ]:
RUN_APP = False    # flip to True to try it; the cell blocks while the app serves
if RUN_APP:
    import os
    os.environ["EE_PROJECT"] = PROJECT   # the form's Project box pre-fills from this
    from gee_animation.gui import build_app
    build_app().launch()   # Colab detected -> Gradio prints a public *.gradio.live link

The link is public while the cell runs — anyone holding it renders under
*your* project. Stop the cell (or the runtime) and it is gone.

## Make it yours

1. **Pre-load your own AOI.** `GEE_DEFAULT_AOI=/path/to/my_aoi.geojson
   gee-animation-gui` starts the app with your area already loaded — the same
   variable the carrot recipe sets.
2. **Add a site preset.** Open `gee_animation/presets.py`: `_SITES` maps a name
   to a footprint GeoJSON and a title, `_EDITIONS` holds the render recipes, and
   the dropdown is their product. Add one `_SITES` entry pointing at your own
   footprint and restart — your site appears in every edition.
3. **From click to config.** Run a thermal render with *pixel grid* ticked in
   the accordion, then find the same two keys (`render.pixel_grid`,
   `render.upscale`) in `config/r12_zoom_lst_pretty_10yr_grid.yaml` — the GUI
   and the site configs speak one language, and notebook 03 teaches it.

*The seam behind it:* `gui.build_app` (`gee_animation/gui.py`) is one function
returning a Gradio `Blocks` app — mount it anywhere Gradio mounts, which is how
the multi-user server hosts this same UI behind a sign-in.